# CIFAR-10 Image Classification Project
## Comparing ANN and CNN on the same dataset

In this notebook I am training two models on CIFAR-10:
- an ANN, which reads each image as a flat vector
- a CNN, which keeps the image shape and learns visual features

The main aim is to see why CNNs usually work better for image classification.


# Problem Statement

Build an image classification model on the CIFAR-10 dataset using:

1. Artificial Neural Network (ANN)
2. Convolutional Neural Network (CNN)

After training both models, compare their accuracy and validation curves.

### CIFAR-10 Classes
Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("Libraries imported successfully")
print("TensorFlow version:", tf.__version__)


# Load Dataset

CIFAR-10 has 60,000 small color images.
- 50,000 images are used for training
- 10,000 images are used for testing
- each image has shape 32 x 32 x 3


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)
print("Classes:", class_names)


## Visualize Sample Images

This step helps check what the dataset actually looks like before training.


In [ ]:
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis("off")
plt.tight_layout()
plt.show()

print("Displayed 10 sample CIFAR-10 images")


# Preprocessing

Pixel values are changed from 0-255 to 0-1. This helps the model train more smoothly.

For the ANN, the image is also flattened from 32 x 32 x 3 into 3072 values.


In [ ]:
x_train_norm = x_train / 255.0
x_test_norm = x_test / 255.0

x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat = x_test_norm.reshape(len(x_test_norm), -1)

print("Normalized training range:", x_train_norm.min(), "to", x_train_norm.max())
print("ANN training shape:", x_train_flat.shape)
print("CNN training shape:", x_train_norm.shape)


# Part 1: ANN Model

The ANN gets a flattened image, so it does not directly understand nearby pixels or image structure.


In [ ]:
ann_model = models.Sequential([
    layers.Dense(512, activation="relu", input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation="relu"),
    layers.Dense(10, activation="softmax")
])

ann_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

ann_model.summary()

ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64,
    verbose=2
)


In [ ]:
ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test, verbose=0)
print("ANN Test Loss:", round(ann_test_loss, 4))
print("ANN Test Accuracy:", round(ann_test_acc, 4))


# Part 2: CNN Model

The CNN keeps the 32 x 32 x 3 image shape. Convolution and pooling layers help it learn edges, shapes, and patterns.


In [ ]:
cnn_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax")
])

cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

cnn_history = cnn_model.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64,
    verbose=2
)


In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print("CNN Test Loss:", round(cnn_test_loss, 4))
print("CNN Test Accuracy:", round(cnn_test_acc, 4))


## Compare Learning Curves

This plot shows which model is learning better on validation data.


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ann_history.history["val_accuracy"], label="ANN validation accuracy")
plt.plot(cnn_history.history["val_accuracy"], label="CNN validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ANN vs CNN Validation Accuracy")
plt.legend()
plt.show()

print("Final ANN validation accuracy:", round(ann_history.history["val_accuracy"][-1], 4))
print("Final CNN validation accuracy:", round(cnn_history.history["val_accuracy"][-1], 4))


# Training Strategy Upgrade: Data Augmentation

Data augmentation creates slightly changed versions of training images. It can help the CNN perform better on new images.


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

aug_cnn_model = models.Sequential([
    data_augmentation,
    layers.Conv2D(32, 3, activation="relu", input_shape=(32, 32, 3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax")
])

aug_cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Augmentation layers added: RandomFlip, RandomRotation, RandomZoom")
print("Augmented CNN is compiled and ready for training")

# Optional training line:
# aug_history = aug_cnn_model.fit(x_train_norm, y_train, epochs=10, validation_split=0.1, batch_size=64, verbose=2)


# Final Comparison Table


In [ ]:
comparison = pd.DataFrame({
    "Model": ["ANN", "CNN"],
    "Test Loss": [ann_test_loss, cnn_test_loss],
    "Test Accuracy": [ann_test_acc, cnn_test_acc]
})

comparison["Test Loss"] = comparison["Test Loss"].round(4)
comparison["Test Accuracy"] = comparison["Test Accuracy"].round(4)
print("Final model comparison")
comparison


# Student Learning Tasks

1. Increase the number of ANN layers and check if accuracy improves.
2. Change CNN filters, for example 32, 64, 128.
3. Increase epochs from 10 to 20.
4. Add EarlyStopping to stop training when validation accuracy stops improving.
5. Train the augmented CNN and compare it with the normal CNN.


## Task 1: Deeper ANN

Adding more Dense layers to see if depth alone helps an ANN on image data.
Spoiler: it usually gives a small bump at best, because flattening still throws away spatial structure.

In [ ]:
deep_ann_model = models.Sequential([
    layers.Dense(1024, activation="relu", input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])

deep_ann_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

deep_ann_model.summary()

deep_ann_history = deep_ann_model.fit(
    x_train_flat, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64,
    verbose=2
)


In [ ]:
deep_ann_test_loss, deep_ann_test_acc = deep_ann_model.evaluate(x_test_flat, y_test, verbose=0)
print("Deep ANN Test Loss:", round(deep_ann_test_loss, 4))
print("Deep ANN Test Accuracy:", round(deep_ann_test_acc, 4))
print("Original ANN Test Accuracy:", round(ann_test_acc, 4))
print("Improvement:", round(deep_ann_test_acc - ann_test_acc, 4))


## Task 2: CNN with Different Filter Sizes

Trying a wider filter progression (64 -> 128 -> 256 instead of 32 -> 64 -> 128) to see how filter count affects accuracy and training time.

In [ ]:
wide_cnn_model = models.Sequential([
    layers.Conv2D(64, (3, 3), activation="relu", input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(256, (3, 3), activation="relu"),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax")
])

wide_cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

wide_cnn_model.summary()

wide_cnn_history = wide_cnn_model.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64,
    verbose=2
)


In [ ]:
wide_cnn_test_loss, wide_cnn_test_acc = wide_cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print("Wide CNN (64-128-256) Test Loss:", round(wide_cnn_test_loss, 4))
print("Wide CNN (64-128-256) Test Accuracy:", round(wide_cnn_test_acc, 4))
print("Original CNN (32-64-128) Test Accuracy:", round(cnn_test_acc, 4))
print("Improvement:", round(wide_cnn_test_acc - cnn_test_acc, 4))


## Task 3: Train CNN for 20 Epochs

Same architecture as the original CNN, just trained for longer to see if accuracy keeps improving or plateaus/overfits.

In [ ]:
cnn_20ep_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax")
])

cnn_20ep_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_20ep_history = cnn_20ep_model.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    verbose=2
)


In [ ]:
cnn_20ep_test_loss, cnn_20ep_test_acc = cnn_20ep_model.evaluate(x_test_norm, y_test, verbose=0)
print("CNN (20 epochs) Test Loss:", round(cnn_20ep_test_loss, 4))
print("CNN (20 epochs) Test Accuracy:", round(cnn_20ep_test_acc, 4))
print("CNN (10 epochs) Test Accuracy:", round(cnn_test_acc, 4))

plt.figure(figsize=(10, 4))
plt.plot(cnn_history.history["val_accuracy"], label="CNN 10 epochs")
plt.plot(cnn_20ep_history.history["val_accuracy"], label="CNN 20 epochs")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Effect of Training Longer (10 vs 20 epochs)")
plt.legend()
plt.show()


## Task 4: CNN with EarlyStopping

Training with `EarlyStopping` so the model stops automatically once validation accuracy stops improving, instead of always running a fixed number of epochs. `restore_best_weights=True` makes sure we keep the best version of the model, not just the last one.

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=3,
    restore_best_weights=True
)

es_cnn_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax")
])

es_cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

es_cnn_history = es_cnn_model.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop],
    verbose=2
)

print("Stopped after", len(es_cnn_history.history["val_accuracy"]), "epochs (out of 20 max)")


In [ ]:
es_cnn_test_loss, es_cnn_test_acc = es_cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print("EarlyStopping CNN Test Loss:", round(es_cnn_test_loss, 4))
print("EarlyStopping CNN Test Accuracy:", round(es_cnn_test_acc, 4))


## Task 5: Train the Augmented CNN

Actually running `.fit()` on the augmented CNN defined earlier (it was only built and compiled before, never trained). We use the same `EarlyStopping` callback so training stops once it stops improving.

In [ ]:
aug_history = aug_cnn_model.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop],
    verbose=2
)


In [ ]:
aug_test_loss, aug_test_acc = aug_cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print("Augmented CNN Test Loss:", round(aug_test_loss, 4))
print("Augmented CNN Test Accuracy:", round(aug_test_acc, 4))
print("Plain CNN Test Accuracy:", round(cnn_test_acc, 4))
print("Improvement from augmentation:", round(aug_test_acc - cnn_test_acc, 4))


## Updated Comparison: All Models

Final side-by-side comparison across the original ANN/CNN and all 5 student task variants.

In [ ]:
full_comparison = pd.DataFrame({
    "Model": [
        "ANN (baseline)",
        "Deep ANN (Task 1)",
        "CNN (baseline)",
        "Wide CNN 64-128-256 (Task 2)",
        "CNN 20 epochs (Task 3)",
        "CNN EarlyStopping (Task 4)",
        "Augmented CNN (Task 5)"
    ],
    "Test Loss": [
        ann_test_loss,
        deep_ann_test_loss,
        cnn_test_loss,
        wide_cnn_test_loss,
        cnn_20ep_test_loss,
        es_cnn_test_loss,
        aug_test_loss
    ],
    "Test Accuracy": [
        ann_test_acc,
        deep_ann_test_acc,
        cnn_test_acc,
        wide_cnn_test_acc,
        cnn_20ep_test_acc,
        es_cnn_test_acc,
        aug_test_acc
    ]
})

full_comparison["Test Loss"] = full_comparison["Test Loss"].round(4)
full_comparison["Test Accuracy"] = full_comparison["Test Accuracy"].round(4)
full_comparison = full_comparison.sort_values("Test Accuracy", ascending=False).reset_index(drop=True)
print("Full comparison across all models, best to worst")
full_comparison


In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(full_comparison["Model"], full_comparison["Test Accuracy"])
plt.xlabel("Test Accuracy")
plt.title("Test Accuracy Across All Models")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


# Conclusion

- ANN can classify images, but it loses the image structure after flattening.
- CNN is better for image data because it learns visual features from the original image shape.
- Dropout, batch normalization, and data augmentation are useful training strategies.
- In most CIFAR-10 experiments, CNN accuracy is better than ANN accuracy.
